# Demo: solving path planning using search

## Instituto Superior Técnico, University of Lisbon

**Problem statement**: given a labirynth comprising a grid of cells with obstacles, find the shortest path from the initial to the final cells.

Author: Rodrigo Ventura (<rodrigo.ventura@tecnico.ulisboa.pt>)

In [ ]:
import search
from PIL import Image
import numpy as np
import skimage
import matplotlib.pyplot as plt
%matplotlib inline

### State space definition

The states are locations on the map and an action is a movement to an adjacent cell not blocked by an obstacle.

The following class defines:
- initialization by loading a map and defining the initial state (0,0) and the goal state (bottom-right cell)
- the actions possible given a state -- all adjacent unblocked cells
- the transitions -- the next state, given a state and an action
- the goal test -- whether a state is at the goal cell
- the path cost -- the sum of the step costs, where each step cost is the euclidean distance among two adjacent cells

In [ ]:
class PathPlanning(search.Problem):

    def __init__(self, image_filename):
        # Loads a map from a given image (white pixels are obstacles, black are free space)
        img = Image.open(image_filename)
        img.load()
        self.map = np.asarray(img)
        # Defines the initial and the goal states
        self.initial = (0, 0)
        self.goal = (self.map.shape[0]-1, self.map.shape[1]-1)
        # Variables for visualization
        self.visited = self.map.copy()
        self.n_gen = 0

    def actions(self, state):
        result = []
        (i, j) = state
        # Ranges over all 8 adjacent cells and 
        # returns the displacements to the unblocked ones
        for di in range(-1, 2):
            for dj in range(-1, 2):
                if (di,dj)!=(0,0):
                    (ni, nj) = (i+di, j+dj)
                    if ni>=0 and ni<self.map.shape[0] and \
                       nj>=0 and nj<self.map.shape[1] and \
                       self.map[ni,nj]==0:
                        result.append((di, dj))
        return result

    def result(self, state, action):
        (i, j) = state
        (di, dj) = action
        # The next state is obtained by adding the displacement (action) to the given state
        (ni, nj) = (i+di, j+dj)
        # Updates variables used for visualization
        self.visited[ni,nj] = 100
        self.n_gen += 1
        return (ni, nj)

    def goal_test(self, state):
        # Returns True if, and only if the given state is at the goal
        return state==self.goal

    def path_cost(self, c, state1, action, state2):
        # Computes the path cost by summing the euclidean distance between
        # the parent (state1) and the current state (state2), to state1 path cost (c)
        return c + np.linalg.norm(np.array(state2) - np.array(state1))
    
    def mark_solution(self, solution):
        # Marks a given solution for visualization
        if solution is not None:
            self.visited[solution.state] = 200
            self.mark_solution(solution.parent)

### Try out some **uninformed** search strategies

Read the comments below for instructions

In [ ]:
# Instantiates a problem given a map filename (map.png)
# You can create your own map and replace the filename below
# Format: white pixels are obstacles and black ones are free space
problem = PathPlanning("map.png")

# Choose here which search strategy to use by uncommenting the corresponding line
solution = search.breadth_first_graph_search(problem)
#solution = search.depth_first_graph_search(problem)
#solution = search.uniform_cost_search(problem)

# Shows some statistics of the search
print("Solution cost:", solution.path_cost)
print("Depth of the solution:", solution.depth)
print("Number of generated nodes:", problem.n_gen)

# Visualization of the solution -- both visited cells and the solution are shown
problem.mark_solution(solution)
plt.figure()
plt.imshow(problem.visited)
plt.show()

### Defines some heuristic functions by extending the state space definition

Two heuristic functions are defined:
- **h_simple** -- euclidean distance from a given cell to the goal
- **h_perfect** -- computes the perfect heuristic from Dijkstra

In [ ]:
class PathPlanningInformed(PathPlanning):
    def __init__(self, *args):
        super().__init__(*args)
        # Pre-computes the best costs from all cells to the goal using Dijkstra
        costs = np.ones_like(problem.map, dtype=np.int32)
        costs[problem.map>0] = -1
        mcp = skimage.graph.MCP_Geometric(costs)
        (self.cum_costs,_) = mcp.find_costs([problem.goal])

    def h_simple(self, node):
        return np.linalg.norm(np.array(node.state) - np.array(self.goal))

    def h_perfect(self, node):
        return self.cum_costs[node.state]        


### Try out some **informed** search strategies

Read the comments below for instructions

In [ ]:
# Instantiates a problem
problem = PathPlanningInformed("map.png")

# Choose here which heuristic function to use by replacing h_simple
solution = search.astar_search(problem, problem.h_simple)

# Shows some statistics of the search
print("Solution cost:", solution.path_cost)
print("Depth of the solution:", solution.depth)
print("Number of generated nodes:", problem.n_gen)

# Visualization of the solution -- both visited cells and the solution are shown
problem.mark_solution(solution)
plt.figure()
plt.imshow(problem.visited)
plt.show()